# Supplier Analysis Notebook

Comprehensive supplier performance analysis and optimization recommendations.

## Contents:
1. Supplier Performance Metrics
2. Supplier Risk Assessment
3. Supplier Cost Analysis
4. Supplier Relationship Analysis
5. Optimization Recommendations

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import sys
sys.path.append('../src')
from data_processor import ProcurementDataProcessor
from analytics import ProcurementAnalytics
from visualizations import ProcurementVisualizations

print("Libraries imported successfully!")

## 1. Load and Prepare Data

In [ ]:
# Initialize data processor and load data
processor = ProcurementDataProcessor()
data = processor.generate_sample_data(num_records=2000)
cleaned_data = processor.clean_data()

print(f"Dataset loaded: {cleaned_data.shape[0]:,} records")
print(f"Date range: {cleaned_data['Date'].min()} to {cleaned_data['Date'].max()}")
print(f"Number of suppliers: {cleaned_data['Supplier'].nunique()}")

## 2. Supplier Performance Metrics

In [ ]:
# Calculate comprehensive supplier metrics
supplier_metrics = cleaned_data.groupby('Supplier').agg({
    'Amount': ['sum', 'mean', 'count', 'std'],
    'Lead_Time': ['mean', 'std', 'min', 'max'],
    'Status': lambda x: (x == 'Completed').mean() * 100,
    'Quantity': 'sum',
    'Unit_Price': 'mean'
}).round(2)

# Flatten column names
supplier_metrics.columns = [
    'Total_Spend', 'Avg_Order_Value', 'Order_Count', 'Spend_Volatility',
    'Avg_Lead_Time', 'Lead_Time_Std', 'Min_Lead_Time', 'Max_Lead_Time',
    'Completion_Rate', 'Total_Quantity', 'Avg_Unit_Price'
]

# Calculate additional metrics
supplier_metrics['Spend_Share'] = (supplier_metrics['Total_Spend'] / supplier_metrics['Total_Spend'].sum() * 100).round(2)
supplier_metrics['Order_Frequency'] = (supplier_metrics['Order_Count'] / 365 * 12).round(2)  # Orders per month
supplier_metrics['Price_Consistency'] = (1 - (supplier_metrics['Spend_Volatility'] / supplier_metrics['Avg_Order_Value'])) * 100
supplier_metrics['Price_Consistency'] = supplier_metrics['Price_Consistency'].clip(0, 100).round(2)

# Calculate performance scores
supplier_metrics['Reliability_Score'] = supplier_metrics['Completion_Rate']
supplier_metrics['Speed_Score'] = np.where(supplier_metrics['Avg_Lead_Time'] > 0, 
                                         100 - ((supplier_metrics['Avg_Lead_Time'] - supplier_metrics['Avg_Lead_Time'].min()) / 
                                               (supplier_metrics['Avg_Lead_Time'].max() - supplier_metrics['Avg_Lead_Time'].min()) * 100), 100)
supplier_metrics['Volume_Score'] = (supplier_metrics['Total_Spend'] / supplier_metrics['Total_Spend'].max() * 100)
supplier_metrics['Consistency_Score'] = supplier_metrics['Price_Consistency']

# Composite performance score
supplier_metrics['Performance_Score'] = (
    0.3 * supplier_metrics['Reliability_Score'] +
    0.25 * supplier_metrics['Speed_Score'] +
    0.25 * supplier_metrics['Volume_Score'] +
    0.2 * supplier_metrics['Consistency_Score']
).round(2)

# Sort by performance score
supplier_metrics = supplier_metrics.sort_values('Performance_Score', ascending=False)

print("Supplier Performance Metrics:")
display(supplier_metrics.head(10))

## 3. Supplier Performance Visualization

In [ ]:
# Create comprehensive supplier dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Total Spend by Supplier', 'Performance Score Distribution',
        'Average Lead Time', 'Completion Rate',
        'Order Frequency', 'Price Consistency'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Total Spend
fig.add_trace(
    go.Bar(x=supplier_metrics.index[:10], y=supplier_metrics['Total_Spend'][:10],
          name='Total Spend', marker_color='#1f77b4'),
    row=1, col=1
)

# Performance Score
fig.add_trace(
    go.Bar(x=supplier_metrics.index[:10], y=supplier_metrics['Performance_Score'][:10],
          name='Performance Score', marker_color='#ff7f0e'),
    row=1, col=2
)

# Average Lead Time
fig.add_trace(
    go.Bar(x=supplier_metrics.index[:10], y=supplier_metrics['Avg_Lead_Time'][:10],
          name='Avg Lead Time', marker_color='#2ca02c'),
    row=2, col=1
)

# Completion Rate
fig.add_trace(
    go.Bar(x=supplier_metrics.index[:10], y=supplier_metrics['Completion_Rate'][:10],
          name='Completion Rate', marker_color='#d62728'),
    row=2, col=2
)

# Order Frequency
fig.add_trace(
    go.Bar(x=supplier_metrics.index[:10], y=supplier_metrics['Order_Frequency'][:10],
          name='Order Frequency', marker_color='#9467bd'),
    row=3, col=1
)

# Price Consistency
fig.add_trace(
    go.Bar(x=supplier_metrics.index[:10], y=supplier_metrics['Price_Consistency'][:10],
          name='Price Consistency', marker_color='#8c564b'),
    row=3, col=2
)

fig.update_layout(
    height=900,
    showlegend=False,
    title_text="Comprehensive Supplier Performance Analysis",
    template='plotly_white'
)

# Update x-axis labels
for i in range(1, 4):
    for j in range(1, 3):
        fig.update_xaxes(tickangle=45, row=i, col=j)

fig.show()

## 4. Supplier Risk Assessment

In [ ]:
# Calculate supplier risk scores
risk_factors = pd.DataFrame(index=supplier_metrics.index)

# Dependency risk (high spend share)
risk_factors['Dependency_Risk'] = np.where(supplier_metrics['Spend_Share'] > 30, 3,
                                         np.where(supplier_metrics['Spend_Share'] > 15, 2, 1))

# Performance risk (low completion rate)
risk_factors['Performance_Risk'] = np.where(supplier_metrics['Completion_Rate'] < 70, 3,
                                          np.where(supplier_metrics['Completion_Rate'] < 85, 2, 1))

# Lead time risk (high average lead time)
lead_time_threshold = supplier_metrics['Avg_Lead_Time'].mean()
risk_factors['Lead_Time_Risk'] = np.where(supplier_metrics['Avg_Lead_Time'] > lead_time_threshold * 1.5, 3,
                                         np.where(supplier_metrics['Avg_Lead_Time'] > lead_time_threshold, 2, 1))

# Volatility risk (high price volatility)
risk_factors['Volatility_Risk'] = np.where(supplier_metrics['Price_Consistency'] < 50, 3,
                                          np.where(supplier_metrics['Price_Consistency'] < 75, 2, 1))

# Concentration risk (low order count but high spend)
risk_factors['Concentration_Risk'] = np.where(
    (supplier_metrics['Order_Count'] < 5) & (supplier_metrics['Total_Spend'] > supplier_metrics['Total_Spend'].median()), 3,
    np.where((supplier_metrics['Order_Count'] < 10) & (supplier_metrics['Total_Spend'] > supplier_metrics['Total_Spend'].median()), 2, 1)
)

# Overall risk score
risk_factors['Overall_Risk_Score'] = risk_factors.sum(axis=1)
risk_factors['Risk_Level'] = np.where(risk_factors['Overall_Risk_Score'] >= 10, 'High',
                                     np.where(risk_factors['Overall_Risk_Score'] >= 6, 'Medium', 'Low'))

# Merge with supplier metrics
supplier_analysis = supplier_metrics.join(risk_factors)

print("Supplier Risk Assessment:")
risk_summary = supplier_analysis.groupby('Risk_Level').size()
for level in ['High', 'Medium', 'Low']:
    if level in risk_summary:
        print(f"  {level} Risk: {risk_summary[level]} suppliers")

print("\nHigh Risk Suppliers:")
high_risk_suppliers = supplier_analysis[supplier_analysis['Risk_Level'] == 'High']
display(high_risk_suppliers[['Total_Spend', 'Completion_Rate', 'Avg_Lead_Time', 'Overall_Risk_Score']])

## 5. Supplier Segmentation

In [ ]:
# Segment suppliers based on performance and importance
def segment_supplier(row):
    # Strategic: High spend, high performance
    if row['Spend_Share'] > 10 and row['Performance_Score'] > 80:
        return 'Strategic'
    # Critical: High spend, low performance
    elif row['Spend_Share'] > 10 and row['Performance_Score'] <= 60:
        return 'Critical'
    # Preferred: Low spend, high performance
    elif row['Spend_Share'] <= 10 and row['Performance_Score'] > 80:
        return 'Preferred'
    # Approved: Low spend, medium performance
    elif row['Spend_Share'] <= 10 and row['Performance_Score'] > 60:
        return 'Approved'
    # Development: Low spend, low performance
    else:
        return 'Development'

supplier_analysis['Segment'] = supplier_analysis.apply(segment_supplier, axis=1)

# Create segmentation visualization
segment_counts = supplier_analysis['Segment'].value_counts()
segment_spend = supplier_analysis.groupby('Segment')['Total_Spend'].sum()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Supplier Count by Segment', 'Total Spend by Segment'),
    specs=[[{"type": "pie"}, {"type": "pie"}]]
)

fig.add_trace(
    go.Pie(labels=segment_counts.index, values=segment_counts.values, name="Count"),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=segment_spend.index, values=segment_spend.values, name="Spend"),
    row=1, col=2
)

fig.update_layout(
    title_text="Supplier Segmentation Analysis",
    height=500,
    template='plotly_white'
)

fig.show()

print("Supplier Segmentation Summary:")
for segment in supplier_analysis['Segment'].unique():
    segment_data = supplier_analysis[supplier_analysis['Segment'] == segment]
    print(f"\n{segment} Segment:")
    print(f"  Count: {len(segment_data)} suppliers")
    print(f"  Total Spend: ${segment_data['Total_Spend'].sum():,.2f}")
    print(f"  Avg Performance: {segment_data['Performance_Score'].mean():.1f}")
    print(f"  Top Suppliers: {', '.join(segment_data.head(3).index.tolist())}")

## 6. Supplier Cost Analysis

In [ ]:
# Analyze cost patterns across suppliers
cost_analysis = cleaned_data.groupby(['Supplier', 'Category']).agg({
    'Amount': ['sum', 'mean', 'count'],
    'Unit_Price': ['mean', 'std'],
    'Quantity': 'sum'
}).round(2)

cost_analysis.columns = ['Total_Spend', 'Avg_Order', 'Order_Count', 'Avg_Unit_Price', 'Price_Std', 'Total_Quantity']
cost_analysis = cost_analysis.reset_index()

# Find price variations for similar categories
category_price_comparison = cost_analysis.groupby('Category').agg({
    'Avg_Unit_Price': ['mean', 'std', 'min', 'max'],
    'Supplier': 'count'
}).round(2)

category_price_comparison.columns = ['Avg_Price', 'Price_Std', 'Min_Price', 'Max_Price', 'Supplier_Count']
category_price_comparison['Price_Range'] = category_price_comparison['Max_Price'] - category_price_comparison['Min_Price']
category_price_comparison['Price_Variation'] = (category_price_comparison['Price_Std'] / category_price_comparison['Avg_Price'] * 100).round(2)

print("Category Price Analysis:")
display(category_price_comparison.sort_values('Price_Variation', ascending=False))

# Identify categories with high price variation
high_variation_categories = category_price_comparison[category_price_comparison['Price_Variation'] > 30]

if not high_variation_categories.empty:
    print("\nCategories with High Price Variation (>30%):")
    for category in high_variation_categories.index:
        category_data = cost_analysis[cost_analysis['Category'] == category]
        print(f"\n{category}:")
        for _, row in category_data.iterrows():
            print(f"  {row['Supplier']}: ${row['Avg_Unit_Price']:.2f} (avg)")

## 7. Optimization Recommendations

In [ ]:
# Generate optimization recommendations
recommendations = []

# 1. Supplier diversification for high dependency
high_dependency = supplier_analysis[supplier_analysis['Spend_Share'] > 20]
for supplier in high_dependency.index:
    recommendations.append({
        'type': 'Supplier Diversification',
        'priority': 'High',
        'supplier': supplier,
        'description': f"Reduce dependency - {supplier} accounts for {high_dependency.loc[supplier, 'Spend_Share']:.1f}% of total spend",
        'action': 'Identify and qualify alternative suppliers',
        'potential_savings': 'Risk reduction'
    })

# 2. Performance improvement for low-performing strategic suppliers
poor_performers = supplier_analysis[
    (supplier_analysis['Segment'] == 'Strategic') & 
    (supplier_analysis['Performance_Score'] < 70)
]
for supplier in poor_performers.index:
    recommendations.append({
        'type': 'Performance Improvement',
        'priority': 'High',
        'supplier': supplier,
        'description': f"Improve performance - strategic supplier with low score ({poor_performers.loc[supplier, 'Performance_Score']:.1f})",
        'action': 'Implement performance improvement plan',
        'potential_savings': 'Service quality improvement'
    })

# 3. Price standardization opportunities
for category in high_variation_categories.index:
    category_data = cost_analysis[cost_analysis['Category'] == category]
    if len(category_data) > 1:
        min_price_supplier = category_data.loc[category_data['Avg_Unit_Price'].idxmin(), 'Supplier']
        max_price_supplier = category_data.loc[category_data['Avg_Unit_Price'].idxmax(), 'Supplier']
        price_diff = category_data['Avg_Unit_Price'].max() - category_data['Avg_Unit_Price'].min()
        
        recommendations.append({
            'type': 'Price Standardization',
            'priority': 'Medium',
            'supplier': f"{category} category",
            'description': f"Price variation of ${price_diff:.2f} in {category}",
            'action': f"Negotiate standardized pricing with {max_price_supplier} or switch to {min_price_supplier}",
            'potential_savings': f'Up to {((price_diff / category_data["Avg_Unit_Price"].max()) * 100):.1f}% cost reduction'
        })

# 4. Lead time optimization
long_lead_time = supplier_analysis[supplier_analysis['Avg_Lead_Time'] > supplier_analysis['Avg_Lead_Time'].mean() * 1.5]
for supplier in long_lead_time.index:
    recommendations.append({
        'type': 'Lead Time Optimization',
        'priority': 'Medium',
        'supplier': supplier,
        'description': f"Long lead time of {long_lead_time.loc[supplier, 'Avg_Lead_Time']:.1f} days",
        'action': 'Review logistics and delivery processes',
        'potential_savings': 'Reduced inventory costs'
    })

# Convert recommendations to DataFrame
recommendations_df = pd.DataFrame(recommendations)

print("Optimization Recommendations:")
display(recommendations_df)

# Summary by priority
print("\nRecommendations by Priority:")
priority_summary = recommendations_df['priority'].value_counts()
for priority in ['High', 'Medium', 'Low']:
    if priority in priority_summary:
        print(f"  {priority}: {priority_summary[priority]} recommendations")

## 8. Export Supplier Analysis Results

In [ ]:
# Create output directory
import os
output_dir = '../data/analysis'
os.makedirs(output_dir, exist_ok=True)

# Export supplier analysis
supplier_analysis.to_csv(f'{output_dir}/supplier_analysis.csv', index=True)
print(f"Supplier analysis exported to: {output_dir}/supplier_analysis.csv")

# Export recommendations
recommendations_df.to_csv(f'{output_dir}/supplier_recommendations.csv', index=False)
print(f"Supplier recommendations exported to: {output_dir}/supplier_recommendations.csv")

# Export cost analysis
cost_analysis.to_csv(f'{output_dir}/supplier_cost_analysis.csv', index=False)
print(f"Cost analysis exported to: {output_dir}/supplier_cost_analysis.csv")

# Create summary report
summary_report = {
    'analysis_date': pd.Timestamp.now().isoformat(),
    'total_suppliers': len(supplier_analysis),
    'high_risk_suppliers': len(supplier_analysis[supplier_analysis['Risk_Level'] == 'High']),
    'strategic_suppliers': len(supplier_analysis[supplier_analysis['Segment'] == 'Strategic']),
    'total_recommendations': len(recommendations_df),
    'high_priority_recommendations': len(recommendations_df[recommendations_df['priority'] == 'High']),
    'top_performer': supplier_analysis.index[0],
    'avg_performance_score': supplier_analysis['Performance_Score'].mean(),
    'total_spend': supplier_analysis['Total_Spend'].sum()
}

import json
with open(f'{output_dir}/supplier_analysis_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2, default=str)

print(f"Summary report exported to: {output_dir}/supplier_analysis_summary.json")
print("\nSupplier analysis completed successfully!")

print(f"\n=== KEY INSIGHTS ===")
print(f"Total Suppliers Analyzed: {summary_report['total_suppliers']}")
print(f"High Risk Suppliers: {summary_report['high_risk_suppliers']}")
print(f"Strategic Suppliers: {summary_report['strategic_suppliers']}")
print(f"Total Recommendations: {summary_report['total_recommendations']}")
print(f"High Priority Actions: {summary_report['high_priority_recommendations']}")
print(f"Top Performing Supplier: {summary_report['top_performer']}")
print(f"Average Performance Score: {summary_report['avg_performance_score']:.1f}")